# Customer Retention & Churn Risk Analytics

Simple end-to-end churn prediction workflow:
1. Data loading & cleaning
2. EDA
3. Feature engineering (encoding, scaling)
4. Model training & comparison (Logistic Regression, KNN, SVC, Decision Tree, Random Forest)
5. Export best model

Dataset: `customer_churn_data.csv` (columns: CustomerID, Age, Gender, Tenure, MonthlyCharges, ContractType, InternetService, TotalCharges, TechSupport, Churn).

## 1. Data Preparation & Exploration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [ ]:
# Load the dataset
df = pd.read_csv('customer_churn_data.csv')
df.head()

In [ ]:
df.shape

In [ ]:
# Check for missing values
df.isnull().sum()

In [ ]:
# Check for duplicate rows
df.duplicated().sum()

In [ ]:
# Handle any missing values in 'InternetService' by filling with an empty string
# (instead of dropping rows, to avoid losing data). This dataset has no nulls here,
# but the step is kept as a safeguard, matching standard cleaning practice.
df['InternetService'] = df['InternetService'].fillna('')

In [ ]:
# Drop CustomerID - it's just an identifier, not useful for prediction
df = df.drop('CustomerID', axis=1)

In [ ]:
# Numerical summary
df.describe()

### Exploratory Data Analysis (EDA)

In [ ]:
# Churn distribution
df['Churn'].value_counts().plot(kind='pie', autopct='%1.1f%%')
plt.title('Churn Distribution')
plt.ylabel('')
plt.show()

In [ ]:
# Histograms for numerical columns
df.hist(figsize=(10, 8))
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap (numerical columns only)
plt.figure(figsize=(8, 6))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

## 2. Feature Engineering & Modeling

### Encoding categorical variables

In [ ]:
# Example: encode Gender and Churn into binary numeric values
df['Gender'] = df['Gender'].map({'Male': 1, 'Female': 0})
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

In [ ]:
# Encode all remaining categorical (text) columns the same simple way
# Keep the fitted encoders so the Streamlit app can reuse the exact same mapping
encoders = {}
categorical_cols = df.select_dtypes(include='object').columns

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

with open('encoders.pickle', 'wb') as f:
    pickle.dump(encoders, f)

df.head()

### Train-Test Split

In [ ]:
X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train.shape, X_test.shape

### Scaling
Fit the scaler only on training data and save it, so the same transformation can be reused on new data later (avoids data leakage).

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

with open('scaler.pickle', 'wb') as f:
    pickle.dump(scaler, f)

### Training Models with GridSearchCV
Simple hyperparameter grids for each model, compared on test accuracy.

In [ ]:
results = {}

In [ ]:
# Logistic Regression
lr_params = {'C': [0.1, 1, 10]}
lr_grid = GridSearchCV(LogisticRegression(max_iter=1000), lr_params, cv=5)
lr_grid.fit(X_train_scaled, y_train)

lr_best = lr_grid.best_estimator_
results['Logistic Regression'] = accuracy_score(y_test, lr_best.predict(X_test_scaled))
print('Best params:', lr_grid.best_params_)
print('Accuracy:', results['Logistic Regression'])

In [ ]:
# K-Nearest Neighbors
knn_params = {'n_neighbors': [3, 5, 7, 9]}
knn_grid = GridSearchCV(KNeighborsClassifier(), knn_params, cv=5)
knn_grid.fit(X_train_scaled, y_train)

knn_best = knn_grid.best_estimator_
results['KNN'] = accuracy_score(y_test, knn_best.predict(X_test_scaled))
print('Best params:', knn_grid.best_params_)
print('Accuracy:', results['KNN'])

In [ ]:
# Support Vector Classifier (probability=True so we can get churn probabilities later)
svc_params = {'C': [0.1, 1, 10], 'kernel': ['linear', 'rbf']}
svc_grid = GridSearchCV(SVC(probability=True), svc_params, cv=5)
svc_grid.fit(X_train_scaled, y_train)

svc_best = svc_grid.best_estimator_
results['SVC'] = accuracy_score(y_test, svc_best.predict(X_test_scaled))
print('Best params:', svc_grid.best_params_)
print('Accuracy:', results['SVC'])

In [ ]:
# Decision Tree
dt_params = {'max_depth': [3, 5, 7, None]}
dt_grid = GridSearchCV(DecisionTreeClassifier(random_state=42), dt_params, cv=5)
dt_grid.fit(X_train_scaled, y_train)

dt_best = dt_grid.best_estimator_
results['Decision Tree'] = accuracy_score(y_test, dt_best.predict(X_test_scaled))
print('Best params:', dt_grid.best_params_)
print('Accuracy:', results['Decision Tree'])

In [ ]:
# Random Forest
rf_params = {'n_estimators': [100, 200], 'max_depth': [5, 10, None]}
rf_grid = GridSearchCV(RandomForestClassifier(random_state=42), rf_params, cv=5)
rf_grid.fit(X_train_scaled, y_train)

rf_best = rf_grid.best_estimator_
results['Random Forest'] = accuracy_score(y_test, rf_best.predict(X_test_scaled))
print('Best params:', rf_grid.best_params_)
print('Accuracy:', results['Random Forest'])

### Model Comparison

In [ ]:
results_df = pd.DataFrame(list(results.items()), columns=['Model', 'Accuracy'])
results_df = results_df.sort_values('Accuracy', ascending=False).reset_index(drop=True)
results_df

### Export Best Model
Based on the reference video, SVC performed best (~94.5% accuracy). Export whichever model scored highest here.

In [ ]:
best_model_name = results_df.iloc[0]['Model']
best_models = {
    'Logistic Regression': lr_best,
    'KNN': knn_best,
    'SVC': svc_best,
    'Decision Tree': dt_best,
    'Random Forest': rf_best
}
best_model = best_models[best_model_name]

with open('model.pickle', 'wb') as f:
    pickle.dump(best_model, f)

print(f'Best model: {best_model_name} — saved as model.pickle')